# Demo — Metadata as a Governance Control (live AWS Glue)

**Trailhead Provisions** runs as a federation of four domain teams on a shared **AWS Glue Data
Catalog**. This short walkthrough demonstrates the **technique** you'll use in the exercise:
treat the catalog as a *control surface* — connect to Glue, run the audit, read a finding, and
**fix it in the catalog**, then re-audit to confirm. The exercise asks you to do the full
audit-and-remediate for the Customer and Marketing domains yourself.

> Runs against live AWS Glue when your lab is provisioned (`lf_backend()` reports `live`),
> otherwise an equivalent local catalog — the technique is identical.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog("trailhead.db")
print("Backend:", lf_backend(), "->", type(gc).__name__)
print("Catalog database:", gc.catalog.database)

## Run the catalog audit
One row per governance gap, read from the Glue catalog across all four domains.

In [ ]:
audit = gc.catalog_audit()
print('Total gaps:', len(audit))
audit

## How to read the audit (two representative findings)

**A table-level gap — `orders` has no owner.** Filter to one table to see its gaps:

In [ ]:
audit[audit["table"] == "orders"]

A missing `owner` means no one is accountable for purchase events. **Fix it in the
catalog** with `set_table_metadata`, which writes the Glue table parameters:

In [ ]:
gc.set_table_metadata("orders", owner="orders-team@trailhead.example")
gc.catalog_audit().pipe(lambda d: d[d["table"] == "orders"])   # gap for orders is gone

**A column-level gap — an untagged PII column.** These are the highest-severity
findings: personal data with no classification is invisible to PII-driven controls.

In [ ]:
audit[audit["gap_type"] == "untagged_column"]

Tag one with `tag_column`, which writes the column's Glue classification parameter:

In [ ]:
gc.tag_column("customer", "email", "PII")
gc.catalog_audit().pipe(lambda d: d[d["gap_type"] == "untagged_column"])

## Gaps aren't only missing tags — domains also disagree
The audit's companion is the semantic-conflict report: places where domains define the same concept differently.

In [ ]:
gc.semantic_conflicts()

Read the **consent** conflict: loyalty and marketing each keep their own consent
flag, and they disagree — acting on the wrong copy is a GDPR violation, surfaced entirely
through metadata.

**Takeaway / your turn:** the catalog is a control surface — audit it, **remediate the gaps in
Glue**, and re-audit to verify. In the exercise you'll do this for the Customer and Marketing
domains and write the reconciliation memo.